In [8]:
import warnings
warnings.filterwarnings('ignore')
import pynamod
import torch
import h5py
from pynamod.geometry.trajectories import H5_Trajectory
import matplotlib.pyplot as plt
import seaborn as sns
import nglview as nv
import numpy as np
import MDAnalysis as mda
from tqdm import tqdm

In [4]:
trajectory_file = ''

In [5]:
def get_cos(cgs):
    coords = []
    ref_inds = np.array([protein.ref_pair.ind for protein in cgs.proteins])
    ref_vecs = cgs.dna.origins[ref_inds].reshape(-1,3)
    ref_vecs = np.diff(ref_vecs,axis=0)
    ref_vecs /= np.linalg.norm(ref_vecs,axis=1,keepdims=True)
    for st in cgs.dna.trajectory:
        coords.append(cgs.dna.origins[ref_inds].astype(np.float64).reshape(-1,3))
    coords = np.stack([coords])[0]    
    test_vecs=coords[:,1:]-coords[:,:-1]
    test_vecs/=np.expand_dims(np.linalg.norm(test_vecs,axis=2),2)
    cosines=np.einsum('kij,kij->ki',ref_vecs.reshape(-1,*ref_vecs.shape),test_vecs )
    return cosines

def get_gyration_radii(cgs,ln):
    pair_names = [pair.pair_name.replace("B'",'') for pair in cgs.dna.pairs_list]
    dna_masses = np.array([nucl_masses[pair_name[0]]+nucl_masses[pair_name[1]] for pair_name in pair_names])
    total_masses = np.hstack([dna_masses] + [protein.masses for protein in cgs.proteins])
    M = total_masses.sum()
    gyr_radii = np.zeros(ln)
    prot_vectors = cgs.proteins[0].ref_vectors.reshape(-1,3).numpy()
    ref_inds = np.array([protein.ref_pair.ind for protein in cgs.proteins])

    for i,st in enumerate(cgs.dna.trajectory):
        ori = cgs.dna.origins.astype(np.float64)
        ref_ori = ori[ref_inds]
        ref_r = cgs.dna.ref_frames[ref_inds].transpose(0,2,1)
        pos = np.vstack([ori.reshape(-1,3),(np.einsum('ij,kjl->kil',prot_vectors,ref_r) + ref_ori).reshape(-1,3)])
        center = np.sum(pos*total_masses.reshape(-1,1),axis=0)/M
        sm = np.sum((pos - center)**2,axis=1)
        radii_square = np.sum(total_masses*sm) / M
        if i != ln:
            gyr_radii[i] = np.sqrt(radii_square)
        

    return gyr_radii

def get_end_end_dist(cgs,ln):
    dist = np.zeros(ln)
    ind1 = [p.ref_pair.ind for p in cgs.proteins[6:9]]
    ind2 = [p.ref_pair.ind for p in cgs.proteins[13:10:-1]]
    for i,st in enumerate(cgs.dna.trajectory):
        ori = cgs.dna.origins.astype(np.float64)
        if i != ln:
            dist[i] = np.linalg.norm(ori[ind1]-ori[ind2],axis=1).mean()

    return dist

def analyze_traj(cgs,traj_file,step):
    cgs.dna.geom_params.trajectory = H5_Trajectory(traj_file,1,len(cgs.dna.pairs_list),mode='r')
    cgs.dna.traj_step = step
    ln = cgs.dna.geom_params.trajectory.get_len()//step

    cosines = get_cos(cgs)
    gyration_radii = get_gyration_radii(cgs,ln)
    dist = get_end_end_dist(cgs,ln)

    return cosines,gyration_radii,dist

In [6]:
nucl_masses = {
    'A':346.2212,
    'T':321.2085,
    'C':322.198,
    'G':362.223
}

In [ ]:
cgs = pynamod.CG_Structure()
file = h5py.File(f'cg_3lz0.h5','r')
cgs.load_from_h5(file)
dna_gen = pynamod.CG_Structure()
dna_gen.build_dna(sequence='atcg'*7)
cgs.append_structures([dna_gen,cgs]*19)
file.close()

In [7]:
step = 10
cosines,gyration_radii,dist = analyze_traj(cgs,trajectory_file,step)
central_dist = get_dist(cgs,trajectory_file,step)

NameError: name 'names' is not defined

In [ ]:
for name in tqdm(names):
    backbones = u_dict[name].atoms[ref_inds]
    plen = polymer.PersistenceLength([backbones])
    plen.run()
    res[name] = plen
    print(plen.results.lp)

In [ ]:
from scipy.optimize import curve_fit
def exp_func(x,lp):
    return(np.exp(-x/lp))

In [9]:
fig,ax=plt.subplots(figsize=(4,3),dpi=200)

val = cosines.mean(axis=0)
popt, pcov = curve_fit(exp_func, np.arange(len(val)), val)
ax.plot(val,color=color)
ax.plot(exp_func(np.arange(len(val)),*popt),'--')

plt.xticks(range(1,19,2))
plt.xlabel('число нуклеосом')
plt.ylabel('ориентационная корреляция')

IndentationError: unexpected indent (12010399.py, line 7)

In [ ]:
fig,ax=plt.subplots(figsize=(8,5),dpi=150)

ax.plot(np.arange(0,gyration_radii.shape[0],1000,int)/10,gyration_radii)

plt.xlabel('Кадры, тысячи')
plt.ylabel('радиус гирации, Å')

In [ ]:
fig,ax=plt.subplots(figsize=(8,5),dpi=150)

val,edges=np.histogram(gyration_radii,bins=50,density=True)
ax.plot(edges[1:])

plt.ylabel('Частота встречаемости')
plt.xlabel('Радиус гирации')

In [ ]:
fig,ax=plt.subplots(figsize=(8,5),dpi=150)

val,edges=np.histogram(central_dist,bins=50,density=True)
ax.plot(edges[1:],val)

plt.ylabel('Частота встречаемости')
plt.xlabel('Среднее расстояние между центральными нуклеосомами')

In [ ]:
fig,ax=plt.subplots(figsize=(4,3),dpi=200)
y = 17.5
def Hooke_law(x, K, b, c):
    return (0.5 * K * (x-b)**2)+c

k_val = []
for name,color in zip(names,pal):
    val,edges = hist_dict[name]
    E = -k*300*N_A*0.001*np.log(val)
    if name == 'no_ptm':
        init = [1,0.1,0]
    else:
        init = [1,1,1]
    popt, pcov = curve_fit(Hooke_law, edges, E,p0=init)
    
    
    ax.plot(edges,E-popt[-1],'.',color=color,label = actual_names[name])
    ax.plot(edges,Hooke_law(edges,*popt)-popt[-1],'--',color=color)
    y -= 0.8
    k_val.append(popt[0])
plt.ylabel('E, КДж/моль')
plt.xlabel('R, Å')

In [ ]:
fig,ax=plt.subplots(figsize=(4,3),dpi=200)
width = 0.5
x = np.arange(len(names))
ax.bar(x,pl_val,width,color=pal)
plt.xticks(x,rotation=45)
ax.set_ylabel('персистентная длина')
ax.set_xticklabels([actual_names[n] for n in names])

In [ ]:
fig,ax=plt.subplots(figsize=(4,3),dpi=200)
width = 0.5
x = np.arange(len(names))
ax.bar(x,k_val,width,color=pal)
plt.xticks(x,rotation=45)
ax.set_ylabel('константа жесткости')
ax.set_xticklabels([actual_names[n] for n in names])

In [ ]:
fig,ax=plt.subplots(figsize=(8,5),dpi=150)
for name,color in zip(names,pal):
    ax.plot(params_dict[name]['dist'][::10],color=color,label=name)

plt.xlabel('Кадры, тысячи')
plt.ylabel('Растояния между концами ДНК')
plt.legend()